# Set thoi gian chay: 1s chay 1 lan
Khi chay thi doc du lieu Redis (Key nhu OG) => Khi co tin hieu thi vao lenh

In [1]:
# Hàm để đặt một lệnh mua
import MetaTrader5 as mt5

def orderSendForex(data):
    
    # Lấy dữ liệu từ dict cua OG
    # datetime = data['Datetime']
    open_price = data['Open']
    close_price = data['Close']
    volume = data['Volume']
    buy_signal = data['Buy_Signal']    
    sell_signal = data['Sell_Signal']
    symbol = data['Symbol'] 
    insertdate = data['Insertdate']

    # vì nó sẽ treo và độ trễ á: Tuan tu => Tat ca deu hoc HFT (n luong)
	
	# Kết nối với MetaTrader 5
    login = 6338576 # XS
    password = '123456Qaz1!'
    server = 'OANDA-Demo-1'    

    # login = 3928 # XS
    # password = '!Od52b03'
    # server = 'XSFintech-DEMO'    

    # if not mt5.initialize(login=48333595, password='9$R#9ht@', server='HFMarketsGlobal-Demo'): # Hotforex Real
    # if not mt5.initialize(login=10138239, password='xiVUPi4J', server='mt5-b2broker-metatrader5.fastmt-b.net:443'): # B2Brocker Demo
    # if not mt5.initialize(login=51524754, password='I2dmjFp5', server='mt5-demo.icmarkets.com'): # ICMarket Demo
    # if not mt5.initialize(login=7174192, password='NttZS95a', server='ICMarketsSC-MT5-2.icmarkets.com'): # ICMarket Real
    # if not mt5.initialize(login=115407068, password='123456Qaz1!', server='mt5trial6.exness.com:443'): # Exness Trial    
    # if not mt5.initialize(login=103466790, password='123456Qaz1!', server='Exness-MT5Real15'): # Exness Real           
    if not mt5.initialize(login=login, password=password, server=server): # XS           
        print("Initialize() failed, error code =", mt5.last_error())
        quit()
    else:
        if(buy_signal == 'True'):
            lot = 0.01  # Số lượng lô mà bạn muốn mua
            # Tinh lot theo 1% balance

            if not mt5.symbol_select(symbol, True): # 
                print(f"Failed to select {symbol}, error code =", mt5.last_error())
                return

            symbol_info = mt5.symbol_info(symbol)
            if symbol_info is None:
                print(f"{symbol} not found")
                return
            
            point = symbol_info.point
            price = mt5.symbol_info_tick(symbol).ask
            deviation = 20  # Độ lệch giá cho phép

            request = {
                "action": mt5.TRADE_ACTION_DEAL,
                "symbol": symbol,
                "volume": lot,
                "type": mt5.ORDER_TYPE_BUY,
                "price": price,
                "tp": price + 400 * point,  # Lệnh chốt lời (Take Profit)
                "sl": price - 200 * point,  # Lệnh dừng lỗ (Stop Loss)
                "deviation": deviation,
                "magic": 234000,
                "comment": "python script open",
                "type_time": mt5.ORDER_TIME_GTC,
                # "type_filling": mt5.ORDER_FILLING_IOC,
                "type_filling": mt5.ORDER_FILLING_FOK,
                
            }

            result = mt5.order_send(request)
            if result.retcode != mt5.TRADE_RETCODE_DONE:
                print("Failed to send order :", result.retcode, result._asdict())
            else:
                print("Order placed BUY successfully!", result)

            # Đóng kết nối với MT5
            mt5.shutdown()

        elif (sell_signal == 'True'):
            lot = 0.01  # Số lượng lô bạn muốn bán
            if not mt5.symbol_select(symbol, True):
                print(f"Failed to select {symbol}, error code =", mt5.last_error())
                return

            symbol_info = mt5.symbol_info(symbol)
            if symbol_info is None:
                print(f"{symbol} not found")
                return
            
            point = symbol_info.point
            price = mt5.symbol_info_tick(symbol).bid  # Sử dụng giá bid cho lệnh bán
            deviation = 20  # Độ lệch giá cho phép

            request = {
                "action": mt5.TRADE_ACTION_DEAL,
                "symbol": symbol,
                "volume": lot,
                "type": mt5.ORDER_TYPE_SELL,
                "price": price,
                "sl": price + 200 * point,  # Lệnh dừng lỗ (Stop Loss) cho lệnh bán
                "tp": price - 400 * point,  # Lệnh chốt lời (Take Profit) cho lệnh bán
                "deviation": deviation,
                "magic": 234000,
                "comment": "python script open",
                "type_time": mt5.ORDER_TIME_GTC,
                # "type_filling": mt5.ORDER_FILLING_IOC,
                "type_filling": mt5.ORDER_FILLING_FOK,
                # ORDER_FILLING_RETURN 
            }

            result = mt5.order_send(request)
            if result.retcode != mt5.TRADE_RETCODE_DONE:
                print("Failed to send order :", result.retcode, result._asdict())
            else:
                print("Order placed SELL successfully!", result)

            # Đóng kết nối với MT5
            mt5.shutdown()


In [2]:
import redis

def entryForex(hash):

	# Xử lý dữ liệu ở đây
	print("Dữ liệu đã được xử lý")
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=6) # Theo ben OG
	# Đọc dữ liệu từ hash
	data = r.hgetall(hash)

	if data:
		# Chuyển đổi dữ liệu từ bytes sang chuỗi nếu cần
		data = {key.decode('utf-8'): value.decode('utf-8') for key, value in data.items()}
		# Xử lý dữ liệu
		# Vao lenh o day
		orderSendForex(data)
		
		# Xóa hash sau khi xử lý
		r.delete(hash)
		print(f"Hash '{hash}' đã được xóa khỏi Redis.")
	else:
		print(f"Không tìm thấy dữ liệu cho hash '{hash}'.")

# Cho 1s chay 1 lan

In [3]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_entryForex():   
	entryForex('Bai tap tai lop 1') # Goi vao ham entryForex de lay du lieu tu Hash Redis

# Lên lịch hàm scan_market để chạy mỗi 1 giây
schedule.every(1).seconds.do(schedule_entryForex)

while True:
	schedule.run_pending()  # Hàm này được gọi liên tục để kiểm tra xem có công việc nào đã đến thời gian cần thực hiện hay không


Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Không tìm thấy dữ liệu cho hash 'Bai tap tai lop 1'.
Dữ liệu đã được xử lý
Khô

KeyError: 'Close'